# Local-Grid Refinement with MODFLOW 6
The purpose of this exercise is to use MODFLOW 6 and local grid refinement to simulate the following simple 3 layer groundwater system.

![lgr_example](../data/lgr_example.png)

## Part I. Imports

In [ ]:
%matplotlib inline
import sys
import os
import pathlib as pl
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import flopy

print(sys.version)
print("python executable: {}".format(sys.executable))
print("numpy version: {}".format(np.__version__))
print("matplotlib version: {}".format(mpl.__version__))
print("flopy version: {}".format(flopy.__version__))

## Part II. Create, Run, and Post-Process MODFLOW 6 Model

In [ ]:
# model info
model_name = "parent"
model_ws = "./model/lgr"

# grid properties
nlay = 3
nrow = 21
ncol = 20
delr = 500.0
delc = 500.0
top = 400.0
botm = [220.0, 200.0, 0.0]

# hydraulic properties
hk0 = 50.0
vk0 = 10.0
hk1 = 0.01
vk1 = 0.01
hk2 = 200.0
vk2 = 20.0

# stresses
well_discharge = -1.5e5
rech = 0.005
rivspd = [[(0, ir, ncol - 1), 320.0, 1.0e5, 318.0] for ir in range(nrow)]

## Part IIA. Create the Parent and Child Grids

Create an integer array (IDOMAIN) where a value of 1 indicates the parent model will be active and 0 where the parent model cell will be replaced by child model cells

In [ ]:
idomain = np.ones((nlay, nrow, ncol), dtype=int)
idomain[0:3, 8:14, 7:13] = 0
idomain[0:3, 7, 8:10] = 0
plt.matshow(idomain[0])

In [ ]:
# create the flopy Lgr object to help define the child model
from flopy.utils.lgrutil import Lgr

ncpp = 3
ncppl = [1, 1, 1]
lgr = Lgr(nlay, nrow, ncol, delr, delc, top, botm, idomain, ncpp, ncppl)

cnlay, cnrow, cncol = lgr.get_shape()
cdelr, cdelc = lgr.get_delr_delc()
ctop, cbotm = lgr.get_top_botm()
cxorigin, cyorigin = lgr.get_lower_left()
cidomain = lgr.get_idomain()
exchangedata = lgr.get_exchange_data(angldegx=True, cdist=True)
nexg = len(exchangedata)

In [ ]:
# create simulation
sim = flopy.mf6.MFSimulation(
    sim_name=model_name, version="mf6", exe_name="mf6", sim_ws=model_ws
)

# create tdis package
tdis = flopy.mf6.ModflowTdis(sim)

# create gwf model
gwf = flopy.mf6.ModflowGwf(
    sim, modelname=model_name, model_nam_file="{}.nam".format(model_name)
)
gwf.name_file.save_flows = True

# create iterative model solution and register the gwf model with it
ims = flopy.mf6.ModflowIms(sim)

# dis
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
    idomain=idomain,
)

# initial conditions
ic = flopy.mf6.ModflowGwfic(gwf, pname="ic", strt=320.0)

# node property flow
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    icelltype=[1, 0, 0],
    k=[hk0, hk1, hk2],
    k33=[vk0, vk1, vk2],
)

# rch
aux = [np.ones((nrow, ncol), dtype=int) * 6]
rch = flopy.mf6.ModflowGwfrcha(
    gwf, recharge=rech, auxiliary=[("iface",)], aux={0: [6]}
)
# riv
riv = flopy.mf6.ModflowGwfriv(gwf, stress_period_data=rivspd)

# output control
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    pname="oc",
    budget_filerecord="{}.cbc".format(model_name),
    head_filerecord="{}.hds".format(model_name),
    headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

# create child gwf model
cmodel_name = "child"
cgwf = flopy.mf6.ModflowGwf(
    sim, modelname=cmodel_name, model_nam_file="{}.nam".format(cmodel_name)
)
cgwf.name_file.save_flows = True
# cgwf.modelgrid.set_coord_info(xoff=cxorigin, yoff=cyorigin)
cdis = flopy.mf6.ModflowGwfdis(
    cgwf,
    nlay=cnlay,
    nrow=cnrow,
    ncol=cncol,
    delr=cdelr,
    delc=cdelc,
    top=ctop,
    botm=cbotm,
    idomain=cidomain,
    xorigin=cxorigin,
    yorigin=cyorigin,
)
cic = flopy.mf6.ModflowGwfic(cgwf, pname="ic", strt=320.0)
cnpf = flopy.mf6.ModflowGwfnpf(
    cgwf,
    save_specific_discharge=True,
    icelltype=[1, 0, 0],
    k=[hk0, hk1, hk2],
    k33=[vk0, vk1, vk2],
)
# rch
aux = [np.ones((nrow, ncol), dtype=int) * 6]
rch = flopy.mf6.ModflowGwfrcha(
    cgwf, recharge=rech, auxiliary=[("iface",)], aux={0: [6]}
)

wi, wj = cgwf.modelgrid.intersect(5000, 5250)
welspd = [[(cnlay - 1, wi, wj), well_discharge]]

wel = flopy.mf6.ModflowGwfwel(cgwf, print_input=True, stress_period_data=welspd)
oc = flopy.mf6.ModflowGwfoc(
    cgwf,
    pname="oc",
    budget_filerecord="{}.cbc".format(cmodel_name),
    head_filerecord="{}.hds".format(cmodel_name),
    headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

# exchange information
gwfe = flopy.mf6.ModflowGwfgwf(
    sim,
    exgtype="GWF6-GWF6",
    nexg=nexg,
    exgmnamea=gwf.name,
    exgmnameb=cgwf.name,
    exchangedata=exchangedata,
    xt3d=True,
    print_flows=True,
    auxiliary=["ANGLDEGX", "CDIST"],
    filename="exchng.gwfgwf",
    # dev_interfacemodel_on=True,
)

sim.write_simulation()
sim.run_simulation()

In [ ]:
# plot grids
fig = plt.figure(figsize=(8, 8))
ax = plt.subplot(1, 1, 1, aspect="equal")
ilay = 0

# plot parent information
pmv = flopy.plot.PlotMapView(model=gwf, layer=ilay)
pmv.plot_grid()

# plot child information
pmv = flopy.plot.PlotMapView(model=cgwf, layer=ilay)
pmv.plot_grid()

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.subplot(1, 1, 1, aspect="equal")
ilay = 2
plot_vectors = True

# plot parent information
pmv = flopy.plot.PlotMapView(model=gwf, layer=ilay)
# pmv.plot_grid()
head = gwf.output.head().get_data()
head = np.ma.masked_where(idomain == 0, head)
vmin = head[ilay].min()
vmax = head[ilay].max()
pmv.plot_array(head, vmin=vmin, vmax=vmax)
if plot_vectors:
    spdis = gwf.output.budget().get_data(text="DATA-SPDIS")[-1]
    qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)
    pmv.plot_vector(qx, qy, normalize=True, color="white")

# plot child information
pmv = flopy.plot.PlotMapView(model=cgwf, layer=ilay)
# pmv.plot_grid()
head = cgwf.output.head().get_data()
head = np.ma.masked_where(head == 1e30, head)
img = pmv.plot_array(head, vmin=vmin, vmax=vmax)
if plot_vectors:
    spdis = cgwf.output.budget().get_data(text="DATA-SPDIS")[-1]
    qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, cgwf)
    pmv.plot_vector(qx, qy, normalize=True, color="white")

ax.set_title(f"Simulation Results for layer {ilay + 1}")
plt.colorbar(img, ax=ax, shrink=0.5)